# End-to-end demo: image -> Module A (what object?) -> B1 (which electrodes?) -> Module C (Dynaphos phosphenes)
Needs `module_a.pt` from the Module A notebook in the working folder. Nothing is trained here; the identifier and the Dynaphos layout are loaded once and kept in memory. The learned planner (B2b) is evaluated in Modules B and E; this demo uses the heuristic planner B1, which works on any new image.

In [ ]:
import os, sys, json, math, time, subprocess, urllib.parse, urllib.request
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageDraw
import yaml, matplotlib.pyplot as plt
from scipy import ndimage
try: import dynaphos
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dynaphos"]); import dynaphos
from dynaphos.simulator import GaussianSimulator
from dynaphos.utils import Map
from dynaphos.cortex_models import get_cortex_coordinates_grid, get_visual_field_coordinates_from_cortex_full

# ---- keep-in-memory helper: an object is built once per kernel; re-running a cell re-uses it ----
_mem = globals().setdefault("_mem", {})
def once(name, key, build, verbose=True):
    if name in globals() and _mem.get(name) == key:
        if verbose: print(f"[memory] {name}: already loaded, nothing recomputed")
        return globals()[name]
    globals()[name] = build(); _mem[name] = key; return globals()[name]


In [ ]:
# ======== settings ========
CKPT_A, N_ELEC, SEED = "module_a.pt", 500, 0
KS = (8, 16, 32, 64)                 # electrode counts to try
SIZE = RES = 256; FOV = 10.0         # image/render size and field of view in degrees
I_MAX = 200.0                        # uA (my choice; verify against safety literature)
DEMO_IMAGE = None                    # e.g. "my_cup.png" to run on a file; None = draw a simple test shape
dev = "cuda" if torch.cuda.is_available() else "cpu"
for k in ("NO_PROXY", "no_proxy"): os.environ[k] = "localhost,127.0.0.1,0.0.0.0"
OUT = "pipeline_demo_out"; os.makedirs(OUT, exist_ok=True); KEY = (SEED, N_ELEC, RES, FOV, I_MAX, CKPT_A)


In [ ]:
# ---------------------------------------------------------------- Module C: frozen Dynaphos + electrode atlas
PARAMS_YAML = f"""
run: {{resolution: [{RES}, {RES}], view_angle: {FOV}, origin: [0, 0], min_angle: 0.001, fps: 35,
      gpu: {0 if torch.cuda.is_available() else 'null'}, print_stats: false, seed: 42, dtype: float32,
      use_gaussian_lut: false, batch_size: 0}}
display: {{screen_resolution: [1920, 1080], screen_diagonal: 13.3, dist_to_screen: 600}}
sampling: {{sampling_method: receptive_fields, RF_size: 0.5, stimulus_scale: 1.e-4}}
cortex_model: {{model: dipole, k: 17.3, a: 0.75, b: 120, alpha: 0.95, dropout_rate: 0.0, noise_scale: 0.4}}
temporal_dynamics: {{trace_increase_rate: 13.95528162, activation_decay_per_second: 0.00012340980408667956, trace_decay_per_second: 0.99949191}}
size: {{size_equation: sqrt, MD: 0.7, I_half: 40, slope_size: 0.08, current_spread: 675.e-6, radius_to_sigma: 0.5}}
brightness_saturation: {{cps_half: 1.057631326853325e-07, slope_brightness: 19152642.500946816}}
thresholding: {{use_threshold: true, rheobase: 23.9e-6, activation_threshold: 9.141886000943878e-08, activation_threshold_sd: 6.715877869833961e-08}}
default_stim: {{relative_stim_duration: 1, pw_default: 170.e-6, freq_default: 300}}
gabor: {{gabor_filtering: false, gamma: 1.5}}
"""   # = dynaphos config/params.yaml; changed: resolution, view_angle, dropout_rate 0 (dropout is applied at render time). Source: Dynaphos repo defaults.

class Phos:
    """Frozen Dynaphos + electrode atlas. render(I_uA) -> phosphene images; only active electrodes are summed (same maths as sim.__call__)."""
    def __init__(s, n_elec, seed):
        s.P = yaml.safe_load(PARAMS_YAML); rng = np.random.default_rng(seed)
        g = 32 if n_elec <= 1000 else 48
        cc = get_cortex_coordinates_grid(s.P["cortex_model"], g, g, x_max=40)       # g x g electrodes per hemisphere on the cortex
        x, y = get_visual_field_coordinates_from_cortex_full(s.P["cortex_model"], cc, rng).cartesian
        inb = (np.abs(x) < FOV / 2) & (np.abs(y) < FOV / 2)
        s.n_total, s.n_in_field = len(x), int(inb.sum())
        keep = rng.choice(np.flatnonzero(inb), min(n_elec, s.n_in_field), replace=False)
        s.sim = GaussianSimulator(s.P, Map(x=x[keep], y=y[keep]), rng)
        s.N, s.dev = s.sim.num_phosphenes, s.sim.data_kwargs["device"]
        co = s.sim.phosphene_maps.flatten(1).argmin(1).cpu().numpy()                 # pixel index of each phosphene centre
        s.xy = np.c_[x[keep], y[keep]]; s.px = np.c_[co % RES, co // RES]            # visual-field (deg) and pixel (col,row) centres
        p, dp, st = s.P["thresholding"], s.P["default_stim"], s.P["run"]["fps"]
        s.k_cur = st / (dp["pw_default"] * dp["freq_default"] * dp["relative_stim_duration"])   # activation -> current factor
        s.thr0 = s.sim.threshold.get().flatten()
        s.thr = ((p["rheobase"] + s.thr0.cpu().numpy() * s.k_cur) * 1e6 * 1.001 + 1e-3).astype(np.float32)   # uA, lowest current that shows a phosphene
        s.usable = s.thr <= I_MAX                                                    # electrodes that can ever be driven
        s.sim.reset(); s.sim.update(torch.full((s.N,), 128e-6, device=s.dev))        # size atlas at a reference current of 128 uA
        s.sigma_deg = s.sim.sigma.get().flatten().cpu().numpy(); s.sigma_px = s.sigma_deg / FOV * (RES - 1)
        s.ecc = np.linalg.norm(s.xy, axis=1)
        b = s.P["brightness_saturation"]; lvl = .7
        s.i70 = float((p["rheobase"] + (b["cps_half"] + math.log(lvl / (1 - lvl)) / b["slope_brightness"]) * s.k_cur) * 1e6)   # uA for 70 % brightness
        sg = torch.tensor(np.maximum(s.sigma_deg, 1.0 * FOV / (RES - 1)), device=s.dev)[:, None, None]
        W = torch.exp(-.5 * (s.sim.phosphene_maps / sg) ** 2).flatten(1); s.W = W / W.sum(1, keepdim=True)   # Gaussian sampling weights per electrode

    def render(s, I, p_drop=0., sigma_thr=0.):
        """I: (B,N) uA tensor (differentiable) -> (B,1,RES,RES). Optional electrode dropout and log-normal threshold jitter."""
        k = max(int((I > 0).sum(1).max()), 1); idx = I.topk(k, 1).indices; out = []
        for b in range(len(I)):
            if sigma_thr > 0: s.sim.threshold.state = (s.thr0 * torch.exp(sigma_thr * torch.randn_like(s.thr0))).view(-1, 1, 1)
            s.sim.reset(); s.sim.update(I[b].clamp_min(1e-3) * 1e-6)                 # clamp: sqrt(0) has an infinite gradient
            i = idx[b]; sg = s.sim.sigma.get().flatten()[i]; br = s.sim.brightness.get().flatten()[i]
            on = (s.sim.activation.get().flatten() > s.sim.threshold.get().flatten())[i].float() * (I[b][i] > 0).float()
            if p_drop > 0: on = on * (torch.rand(k, device=I.device) > p_drop).float()
            g = torch.exp(-.5 * (s.sim.phosphene_maps[i] / sg.clamp_min(1e-22)[:, None, None]) ** 2)
            out.append((on * br)[:, None, None].mul(g).sum(0))
        s.sim.threshold.state = s.thr0.view(-1, 1, 1)
        return torch.stack(out).clamp(0, 1)[:, None]

def save_atlas(C):
    np.savez(f"{OUT}/atlas_s{SEED}.npz", xy_deg=C.xy, px=C.px, sigma_deg=C.sigma_deg, thr_uA=C.thr, usable=C.usable, i70_uA=C.i70)

def check_atlas(C):
    f = f"{OUT}/atlas_s{SEED}.npz"
    if not os.path.exists(f): print("note: no atlas from the Module C notebook yet (run it once). The layout is rebuilt here from the same settings."); return
    a = np.load(f)
    assert a["xy_deg"].shape == C.xy.shape and np.allclose(a["xy_deg"], C.xy) and np.allclose(a["thr_uA"], C.thr, rtol=1e-3), \
        "this layout differs from the atlas written by the Module C notebook: settings or seed changed. Re-run Module C."
    print("layout matches the atlas written by the Module C notebook")


In [ ]:
C = once("C", KEY, lambda: Phos(N_ELEC, SEED)); print(f"Module C ready: {C.N} electrodes, {C.usable.sum()} usable (threshold <= {I_MAX:.0f} uA)")

In [ ]:
def gabor_bank(sig=2., lam=4., n=8, ks=15):
    r = torch.arange(ks, dtype=torch.float32) - ks // 2; yy, xx = torch.meshgrid(r, r, indexing="ij"); g = torch.exp(-(xx ** 2 + yy ** 2) / (2 * sig ** 2)); w = []
    for th in np.arange(n) * np.pi / n:
        ph = 2 * np.pi * (xx * np.cos(th) + yy * np.sin(th)) / lam
        ev = g * torch.cos(ph); ev = ev - g * ev.sum() / g.sum()                       # even part with the DC removed
        w += [ev, g * torch.sin(ph)]
    return torch.stack(w)[:, None].to(dev)                                            # (16,1,ks,ks): 8 orientations x (even, odd)

def blur(x, sig):
    r = int(3 * sig) + 1; k = torch.exp(-.5 * (torch.arange(-r, r + 1, device=x.device) / sig) ** 2); k = (k / k.sum()).to(x.dtype)
    return F.conv2d(F.conv2d(x, k.view(1, 1, 1, -1), padding=(0, r)), k.view(1, 1, -1, 1), padding=(r, 0))

def contour_map(x, GB):                                                               # x (B,1,256,256) -> C = sqrt(even^2 + odd^2), max over orientation
    r = F.conv2d(x, GB, padding=GB.shape[-1] // 2).view(len(x), 8, 2, SIZE, SIZE)
    c = r.pow(2).sum(2).sqrt().amax(1, keepdim=True); return c / c.amax((2, 3), keepdim=True).clamp_min(1e-6)


GB = once("GB", KEY, gabor_bank, verbose=False)

In [ ]:
# ---------------------------------------------------------------- reference arms, B1, planner network, constraints
def pick(xy, score, K, n0=120, r0=.15, floor=.1):
    """Greedy best-score selection with exclusion radius r0 (deg) and a 10 % floor, then homogeneous thinning to K (paper's density reduction)."""
    if score.max() <= 0: return np.array([], int)
    s = np.where(score >= floor * score.max(), score, -1.); ch = []
    while len(ch) < n0 and s.max() > 0:
        e = int(s.argmax()); ch.append(e); s[np.linalg.norm(xy - xy[e], axis=1) < r0] = -1
    ch = np.array(ch); Dm = np.linalg.norm(xy[ch][:, None] - xy[ch][None], axis=-1); alive = np.ones(len(ch), bool)
    while alive.sum() > K:
        i, j = np.unravel_index(np.where(alive[:, None] & alive[None] & ~np.eye(len(ch), dtype=bool), Dm, np.inf).argmin(), Dm.shape)
        alive[i if score[ch[i]] < score[ch[j]] else j] = False                        # drop the weaker of the closest pair
    return ch[alive]

def plan_b1(C, score, K):
    ch = pick(C.xy, score * C.usable, K); I = np.zeros(C.N, np.float32); I[ch] = np.clip(C.i70, C.thr[ch], I_MAX); return I

def plan_r1(C, K, rng):
    ch = rng.choice(np.flatnonzero(C.usable), K, replace=False); I = np.zeros(C.N, np.float32); I[ch] = rng.uniform(C.thr[ch], I_MAX); return I



In [ ]:
# ---------- Module A: load the trained identifier (module_a.pt from the Module A notebook) ----------
from PIL import ImageFilter
def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
class Net(nn.Module):
    def __init__(s, n):
        super().__init__()
        s.f = nn.Sequential(block(1, 32), block(32, 32), nn.MaxPool2d(2), block(32, 64), block(64, 64), nn.MaxPool2d(2), block(64, 128), block(128, 128), nn.MaxPool2d(2), block(128, 256))
        s.head = nn.Sequential(nn.Dropout(.3), nn.Linear(256, n))
    def forward(s, x): return s.head(s.f(x).mean((2, 3)))
def load_a():
    ck = torch.load(CKPT_A, map_location=dev); n = Net(len(ck["classes"])).to(dev); n.load_state_dict(ck["state_dict"]); return n.eval(), ck["classes"]
netA = once("netA", KEY, lambda: load_a()[0])
CLASSES = torch.load(CKPT_A, map_location="cpu")["classes"]

def ink_square(img):
    """drawing / photo / file -> square float array, ink = 1 on black, cropped to the drawing (None if blank)."""
    if isinstance(img, str): img = Image.open(img)
    if isinstance(img, dict): img = img["composite"]                 # Sketchpad gives a dict; composite is None before drawing starts
    if img is None: return None
    im = Image.fromarray(img) if isinstance(img, np.ndarray) else img
    if im.mode == "RGBA": im = Image.alpha_composite(Image.new("RGBA", im.size, "white"), im)
    g = np.asarray(im.convert("L"), np.float32); dv = np.abs(g - np.median(g)); mx = dv.max()
    if mx < 20: return None
    ink = np.where(dv > .3 * mx, dv / mx, 0); ys, xs = np.where(ink > 0)
    ink = ink[ys.min():ys.max() + 1, xs.min():xs.max() + 1]; s = max(ink.shape); sq = np.zeros((s, s), np.float32)
    oy, ox = (s - ink.shape[0]) // 2, (s - ink.shape[1]) // 2; sq[oy:oy + ink.shape[0], ox:ox + ink.shape[1]] = ink
    return sq

def to28(sq):                                                        # input format of Module A
    t = Image.fromarray((sq * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(2 * (len(sq) // 56) + 1))
    a = np.asarray(t.resize((28, 28), Image.BOX), np.float32); return a / max(a.max(), 1)

def to256(sq):                                                       # input format of Module B: drawing spans ~86 % of the 10-degree field, ~3 px strokes
    t = Image.fromarray((sq * 255).astype(np.uint8)).resize((220, 220), Image.BILINEAR).filter(ImageFilter.MaxFilter(3))
    c = np.zeros((SIZE, SIZE), np.float32); c[18:238, 18:238] = np.asarray(t, np.float32) / 255; return c

@torch.no_grad()
def identify(sq, k=5):
    p = netA(torch.tensor(to28(sq), device=dev)[None, None]).softmax(1)[0].cpu(); top = p.topk(k)
    return [(CLASSES[i], float(v)) for v, i in zip(top.values, top.indices)]


In [ ]:
# ---------- the funnel: A's output goes into B (B1 heuristic), B's electrodes go into C (Dynaphos) ----------
def plan_image(img256, K):
    c = contour_map(torch.tensor(img256, device=dev)[None, None], GB); score = (c.flatten(1) @ C.W.T).cpu().numpy()[0]
    return plan_b1(C, score, K)

def run(img, Ks=KS, save=f"{OUT}/a_to_b_result.png"):
    sq = ink_square(img)
    if sq is None: print("nothing drawn"); return None
    top = identify(sq); label = top[0][0]; img256 = to256(sq); res = {"label": label, "top5": top, "by_K": {}}
    fig, ax = plt.subplots(1, len(Ks) + 1, figsize=(3 * (len(Ks) + 1), 3.4)); ax[0].imshow(img256, cmap="gray")
    ax[0].set_title("A: " + "\n".join(f"{n} {p:.0%}" for n, p in top[:3]), fontsize=8)
    for a, K in zip(ax[1:], Ks):
        I = plan_image(img256, K); sel = np.flatnonzero(I); P = C.render(torch.tensor(I, device=dev)[None])[0, 0].cpu().numpy()
        res["by_K"][K] = dict(n_electrodes=len(sel), electrodes=sel, xy_deg=C.xy[sel], current_uA=I[sel], total_mA=float(I.sum() / 1000), phosphenes=P)
        a.imshow(F.max_pool2d(torch.tensor(P)[None, None], 3, 1, 1)[0, 0], cmap="gray", vmin=0, vmax=1)   # 3x3 dilation for display only
        a.set_title(f"B (K={K}): {len(sel)} electrodes\n{I.sum() / 1000:.1f} mA total", fontsize=8)
    for a in ax: a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.savefig(save, dpi=110); plt.show(); plt.close(fig)
    print(f"A says: {label} ({top[0][1]:.0%}). " + " | ".join(f"K={K}: {r['n_electrodes']} electrodes" for K, r in res["by_K"].items()) + f"   -> {save}")
    return res

if DEMO_IMAGE is None:                                               # a plain test drawing: a rectangle with a handle
    t = Image.new("L", (300, 300), 255); dr = ImageDraw.Draw(t); dr.rectangle([70, 90, 200, 230], outline=0, width=6); dr.arc([170, 120, 250, 200], -90, 90, fill=0, width=6)
    DEMO_IMAGE = t
res = run(DEMO_IMAGE)


### Optional app

In [ ]:
# ---------- OPTIONAL app: draw / camera / upload. Run this cell only when you want it. ----------
def app_fn(img, K):
    r = run(img, Ks=(int(K),))
    if r is None: return {"(nothing drawn yet)": 1.0}, None, "no drawing found"
    b = r["by_K"][int(K)]
    return ({n: p for n, p in r["top5"]}, Image.open(f"{OUT}/a_to_b_result.png"),
            f"Object: {r['label']}\nElectrodes selected: {b['n_electrodes']} of {C.N} (requested {int(K)})\n"
            f"Total current: {b['total_mA']:.2f} mA | per electrode: {b['current_uA'].min():.0f}-{b['current_uA'].max():.0f} uA\nElectrode ids: {b['electrodes'].tolist()}")
LAUNCH_APP = True
if LAUNCH_APP:
    try: import gradio as gr
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"]); import gradio as gr
    outs = lambda: [gr.Label(num_top_classes=5), gr.Image(label="A -> B -> C"), gr.Textbox(label="Module B output", lines=5)]
    slider = lambda: gr.Slider(4, 64, 16, step=1, label="electrodes K")
    gr.close_all()
    ui = gr.TabbedInterface([gr.Interface(app_fn, [gr.Sketchpad(), slider()], outs()), gr.Interface(app_fn, [gr.Image(sources=["webcam", "upload"]), slider()], outs())], ["Draw", "Camera / Upload"])
    if "google.colab" in sys.modules: ui.launch(share=True)
    else: ui.launch(server_name="127.0.0.1", inbrowser=True)
